# Sports Reference CFB Scraper

This notebook scrapes College Football data from Sports Reference, including annual stats (passing, rushing, etc.) and the master player index.

In [1]:
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import time
import re
import os
import sys
import string

In [ ]:
# Configuration
YEARS = range(2015, 2026)  # 2015 to 2025 (end is exclusive in range)
CATEGORIES = ['passing', 'rushing', 'receiving', 'kicking', 'punting']
BASE_URL = "https://www.sports-reference.com/cfb/years/{year}-{category}.html"
PLAYER_INDEX_URL = "https://www.sports-reference.com/cfb/players/{letter}-index.html"

# Set base directory for all data operations
# Using relative paths from workspace root
BASE_DATA_DIR = os.path.join(os.path.dirname(os.path.dirname(__file__)), "data")
if not os.path.exists(BASE_DATA_DIR):
    BASE_DATA_DIR = "data"  # Fallback if running from different directory

print(f"Base data directory: {os.path.abspath(BASE_DATA_DIR)}")

In [2]:
def get_soup(url):
    """Fetches URL and returns BeautifulSoup object with error handling."""
    try:
        response = requests.get(url)
        response.raise_for_status()
        return BeautifulSoup(response.content, 'lxml')
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

def extract_table_from_comments(soup, table_id):
    """
    Sports Reference often hides the main data table inside HTML comments.
    This function finds comments, checks for the table ID, and returns the HTML string.
    """
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        if table_id in comment:
            try:
                comment_soup = BeautifulSoup(comment, 'lxml')
                table = comment_soup.find('table', id=table_id)
                if table:
                    return str(table)
            except:
                continue
    return None

In [ ]:
def scrape_stats():
    """Scrapes stats for defined categories and years."""
    
    # Ensure base directory exists
    base_dir = os.path.join(BASE_DATA_DIR, "Stats")
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)

    for category in CATEGORIES:
        category_lower = category.lower()
        category_dir = None
        if os.path.exists(base_dir):
            for d in os.listdir(base_dir):
                if os.path.isdir(os.path.join(base_dir, d)) and d.lower() == category_lower:
                    category_dir = os.path.join(base_dir, d)
                    break
        if not category_dir:
            category_dir = os.path.join(base_dir, category)
            os.makedirs(category_dir)

        print(f"\n--- Starting scrape for category: {category.upper()} ---")
        
        # Check existing files to skip already scraped years
        existing_years = set()
        if os.path.exists(category_dir):
            for file in os.listdir(category_dir):
                if file.endswith(f"_{category}.csv"):
                    year_str = file.split('_')[0]
                    try:
                        existing_years.add(int(year_str))
                    except ValueError:
                        pass
        
        for year in YEARS:
            if year in existing_years:
                print(f"Skipping {year} {category}, already exists.")
                continue
            
            url = BASE_URL.format(year=year, category=category)
            print(f"Scraping {year} {category}...", end=" ")
            
            soup = get_soup(url)
            if not soup:
                continue

            try:
                # Find the stats table
                table = soup.find('table')
                if table:
                    # Parse rows
                    data = []
                    tbody = table.find('tbody')
                    if tbody:
                        rows = tbody.find_all('tr')
                    else:
                        rows = table.find_all('tr')[1:]  # skip header if no tbody
                    
                    for row in rows:
                        cells = row.find_all('td')
                        if not cells:
                            continue
                        row_data = {}
                        for cell in cells:
                            data_stat = cell.get('data-stat', '')
                            if data_stat == 'name_display':
                                a = cell.find('a')
                                if a and 'href' in a.attrs:
                                    link = a['href']
                                    name = a.get_text(strip=True).rstrip('*')
                                    player_id = cell.get('data-append-csv', '')
                                    row_data['Player'] = name
                                    row_data['Player_ID'] = player_id
                                    row_data['Player_Link'] = link
                                else:
                                    row_data['Player'] = cell.get_text(strip=True).rstrip('*')
                                    row_data['Player_ID'] = ''
                                    row_data['Player_Link'] = ''
                            else:
                                row_data[data_stat] = cell.get_text(strip=True)
                        data.append(row_data)
                    
                    df = pd.DataFrame(data)
                    
                    # Clean up: Sports Reference repeats headers every ~20 rows
                    if 'Rk' in df.columns:
                        df = df[df['Rk'] != 'Rk']
                    
                    df['Year'] = year
                    
                    # Save individual file
                    filename = os.path.join(category_dir, f"{year}_{category}.csv")
                    df.to_csv(filename, index=False)
                    print(f"Saved to {filename}")
                else:
                    print("No table found on page.")
            except Exception as e:
                print(f"Error parsing page: {e}")

            # Rate limiting to avoid overwhelming the server
            time.sleep(5)

In [ ]:
def scrape_player_index():
    """Scrapes player metadata from the A-Z index pages."""
    print(f"\n--- Scraping Player Index (A-Z) ---")
    
    players_data = []
    letters = string.ascii_lowercase
    
    for letter in letters:
        url = PLAYER_INDEX_URL.format(letter=letter)
        print(f"Scraping letter {letter.upper()}...", end=" ")
        
        soup = get_soup(url)
        if not soup:
            time.sleep(4)
            continue

        # Find the container if it exists, otherwise search all paragraphs
        container = soup.find('div', id='div_players')
        paragraphs = container.find_all('p') if container else soup.find_all('p')

        count = 0
        for p in paragraphs:
            links = p.find_all('a')
            if len(links) >= 2:
                player_link = links[0]
                school_link = links[1]
                
                # Extract Name
                player_name = player_link.get_text(strip=True)
                
                # Extract ID from href (e.g., /cfb/players/daylin-caamano-1.html)
                href = player_link.get('href', '')
                # Regex to capture the part between last slash and .html
                id_match = re.search(r'/([^/]+)\\.html$', href)
                player_id = id_match.group(1) if id_match else None
                
                # Extract School
                school_name = school_link.get_text(strip=True)
                
                # Extract Years (text inside parentheses)
                full_text = p.get_text()
                years_match = re.search(r'\\((\\d{4}-\\d{4})\\)', full_text)
                years_played = years_match.group(1) if years_match else None
                
                if player_id:
                    players_data.append({
                        'Player Name': player_name,
                        'Player ID': player_id,
                        'School': school_name,
                        'Years Played': years_played
                    })
                    count += 1
        
        print(f"Found {count} players.")
        time.sleep(4)

    if players_data:
        df = pd.DataFrame(players_data)
        output_file = os.path.join(BASE_DATA_DIR, "player_index.csv")
        df.to_csv(output_file, index=False)
        print(f"Successfully scraped {len(df)} players. Saved to {output_file}.")
    else:
        print("No player data found.")

def compile_player_stats():
    """
    Compiles player information from all scraped stat CSV files.
    For each unique player ID, gathers:
    - Player name
    - Years played (years with stats)
    - All teams played for
    """
    print(f"\n--- Compiling Player Stats from CSV Files ---")
    
    players_data = {}  # {player_id: {name, years, teams}}
    base_dir = os.path.join(BASE_DATA_DIR, "Stats")
    
    # Iterate through all category directories
    for category_dir in os.listdir(base_dir):
        category_path = os.path.join(base_dir, category_dir)
        if not os.path.isdir(category_path):
            continue
        
        print(f"Processing category: {category_dir}...", end=" ")
        
        # Iterate through all CSV files in the category
        csv_count = 0
        for csv_file in os.listdir(category_path):
            if not csv_file.endswith('.csv'):
                continue
            
            csv_path = os.path.join(category_path, csv_file)
            try:
                df = pd.read_csv(csv_path)
                csv_count += 1
                
                # Process each row to extract player data
                for idx, row in df.iterrows():
                    player_id = row.get('Player_ID', '')
                    if not player_id or pd.isna(player_id):
                        continue
                    
                    player_name = row.get('Player', 'Unknown')
                    year = row.get('Year', '')
                    
                    # Try different team column names
                    team = row.get('team_name_abbr') or row.get('Team') or row.get('team_name')
                    if pd.isna(team):
                        team = 'Unknown'
                    
                    if player_id not in players_data:
                        players_data[player_id] = {
                            'Player_Name': player_name,
                            'Years': set(),
                            'Teams': set()
                        }
                    
                    # Add year and team
                    if year:
                        players_data[player_id]['Years'].add(int(year))
                    if team and team != 'Unknown':
                        players_data[player_id]['Teams'].add(team)
            except Exception as e:
                print(f"Error reading {csv_path}: {e}")
        
        print(f"Processed {csv_count} files.")
    
    # Convert sets to sorted lists and create DataFrame
    player_list = []
    for player_id, data in players_data.items():
        years_list = sorted(list(data['Years']))
        teams_list = sorted(list(data['Teams']))
        
        # Determine career window
        career_start = min(years_list) if years_list else None
        career_end = max(years_list) if years_list else None
        career_window = f"{career_start}-{career_end}" if career_start and career_end else None
        
        player_list.append({
            'Player_ID': player_id,
            'Player_Name': data['Player_Name'],
            'Career_Window': career_window,
            'Years': ', '.join(map(str, years_list)),
            'Teams': ', '.join(teams_list),
            'Num_Years': len(years_list),
            'Num_Teams': len(teams_list)
        })
    
    # Create and save DataFrame
    compiled_df = pd.DataFrame(player_list)
    compiled_df = compiled_df.sort_values('Player_Name')
    
    output_file = os.path.join(BASE_DATA_DIR, "compiled_player_stats.csv")
    compiled_df.to_csv(output_file, index=False)
    print(f"\nSuccessfully compiled {len(compiled_df)} unique players.")
    print(f"Saved to {output_file}.")
    
    return compiled_df

In [46]:
# Run Stats Scraper
scrape_stats()


--- Starting scrape for category: PASSING ---
Scraping 2015 passing... Saved to Stats\passing\2015_passing.csv
Scraping 2016 passing... Saved to Stats\passing\2016_passing.csv
Skipping 2017 passing, already exists.
Skipping 2018 passing, already exists.
Skipping 2019 passing, already exists.
Skipping 2020 passing, already exists.
Skipping 2021 passing, already exists.
Skipping 2022 passing, already exists.
Skipping 2023 passing, already exists.
Skipping 2024 passing, already exists.
Skipping 2025 passing, already exists.

--- Starting scrape for category: RUSHING ---
Skipping 2015 rushing, already exists.
Skipping 2016 rushing, already exists.
Skipping 2017 rushing, already exists.
Skipping 2018 rushing, already exists.
Skipping 2019 rushing, already exists.
Skipping 2020 rushing, already exists.
Skipping 2021 rushing, already exists.
Skipping 2022 rushing, already exists.
Skipping 2023 rushing, already exists.
Skipping 2024 rushing, already exists.
Skipping 2025 rushing, already exis

In [ ]:
import openpyxl

def scrape_school_url_mapping(html_file=None):
    """
    Scrapes the school list HTML file to extract the correct URL patterns for each school.
    Returns a mapping of school_name -> url_name
    
    Args:
        html_file: Path to HTML file. If None, uses BASE_DATA_DIR/HTML/school_list.txt
    """
    if html_file is None:
        html_file = os.path.join(BASE_DATA_DIR, "HTML", "school_list.txt")
    
    school_url_map = {}
    
    try:
        with open(html_file, 'r', encoding='utf-8') as f:
            html_content = f.read()
        
        soup = BeautifulSoup(html_content, 'lxml')
        
        # Find all rows in the table
        rows = soup.find_all('tr')
        
        for row in rows:
            # Find the school_name cell
            school_cell = row.find('td', {'data-stat': 'school_name'})
            if school_cell:
                link = school_cell.find('a')
                if link and 'href' in link.attrs:
                    href = link['href']
                    # Extract URL name from href: /cfb/schools/air-force/ -> air-force
                    url_name = href.split('/cfb/schools/')[1].rstrip('/')
                    
                    school_name = link.get_text(strip=True)
                    school_url_map[school_name] = url_name
                    
                    # Also add variations (without spaces, etc.) as fallback
                    school_url_map[school_name.replace(' ', '_')] = url_name
        
        print(f"Loaded {len(school_url_map)} school URL mappings from {html_file}")
        return school_url_map
    
    except Exception as e:
        print(f"Error scraping school URL mapping: {e}")
        return {}

def get_school_name_to_url(school_name, school_url_map=None):
    """
    Converts school name to URL format using the mapping from school_list.txt.
    Falls back to simple conversion if not found in map.
    
    Args:
        school_name: The name of the school
        school_url_map: Dictionary mapping school names to URL patterns
    
    Returns:
        URL-safe name for the school
    """
    if school_url_map and school_name in school_url_map:
        return school_url_map[school_name]
    
    # Fallback: simple conversion
    url_name = school_name.lower().strip()
    url_name = url_name.replace(' ', '-')
    url_name = ''.join(c if c.isalnum() or c == '-' else '' for c in url_name)
    while '--' in url_name:
        url_name = url_name.replace('--', '-')
    return url_name.strip('-')

def load_teams_from_excel(excel_path=None, min_year=2015, max_year=2025):
    """
    Loads teams from the Excel file that were active during the specified years.
    Returns a list of tuples: (school_name, url_name)
    
    Args:
        excel_path: Path to Excel file. If None, uses BASE_DATA_DIR/../all_teams.xlsx
    """
    if excel_path is None:
        excel_path = os.path.join(os.path.dirname(BASE_DATA_DIR), "all_teams.xlsx")
    
    try:
        wb = openpyxl.load_workbook(excel_path)
        ws = wb.active
        
        teams = []
        
        # Skip header row, iterate through data rows
        for row in ws.iter_rows(min_row=2, values_only=False):
            # Get school name from column B (index 1)
            school_cell = row[1]
            school_name = school_cell.value
            
            if not school_name:
                continue
            
            # Get year_min from column C (index 2) and year_max from column D (index 3)
            year_min_cell = row[2]
            year_max_cell = row[3]
            
            try:
                year_min = int(year_min_cell.value) if year_min_cell.value else 9999
                year_max = int(year_max_cell.value) if year_max_cell.value else 0
            except (ValueError, TypeError):
                continue
            
            # Check if school was active during our target years
            if year_max >= min_year and year_min <= max_year:
                url_name = get_school_name_to_url(school_name)
                teams.append((school_name, url_name, year_min, year_max))
        
        return sorted(teams, key=lambda x: x[0])  # Sort by school name
    
    except Exception as e:
        print(f"Error loading Excel file: {e}")
        return []

In [ ]:

def scrape_defense_stats(excel_path=None, missing_only=True):
    """
    Scrapes defensive stats for all schools that played during 2015-2025.
    Uses team-by-team scraping from individual school pages.
    
    Args:
        excel_path: Path to Excel file with team information. If None, uses default from BASE_DATA_DIR
        missing_only: If True, only scrapes schools/years that are missing data
    """
    
    min_year = 2015
    max_year = 2025
    target_years = list(range(min_year, max_year + 1))
    
    # Load school URL mapping from HTML
    print("Loading school URL mappings...")
    school_url_map = scrape_school_url_mapping()
    print()
    
    # Load teams from Excel
    teams = load_teams_from_excel(excel_path, min_year, max_year)
    defense_dir = os.path.join(BASE_DATA_DIR, "Stats", "defense")
    
    if not os.path.exists(defense_dir):
        os.makedirs(defense_dir)
    
    # Update team URLs using the mapping
    teams_with_correct_urls = []
    for school_name, url_name, year_min, year_max in teams:
        correct_url_name = get_school_name_to_url(school_name, school_url_map)
        teams_with_correct_urls.append((school_name, correct_url_name, year_min, year_max))
    
    # Get list of missing data if only scraping missing
    if missing_only:
        print("Checking for missing data...\n")
        missing_data = {}
        for school_name, url_name, year_min, year_max in teams_with_correct_urls:
            active_years = [y for y in target_years if year_min <= y <= year_max]
            if not active_years:
                continue
            
            missing_years = []
            for year in active_years:
                file_path = os.path.join(defense_dir, f"{school_name.replace(' ', '_')}_{year}_defense.csv")
                if not os.path.exists(file_path):
                    missing_years.append(year)
            
            if missing_years:
                missing_data[school_name] = (url_name, year_min, year_max, missing_years)
        
        teams_to_scrape = missing_data
        print(f"Found {len(teams_to_scrape)} schools with missing data.\n")
    else:
        teams_to_scrape = {school: (url, y_min, y_max, [y for y in target_years if y_min <= y <= y_max]) 
                          for school, url, y_min, y_max in teams_with_correct_urls}
        print(f"Will attempt to scrape all {len(teams_to_scrape)} schools.\n")
    
    print(f"--- Starting defense stats scrape (missing_only={missing_only}) ---\n")
    
    scraped_count = 0
    for idx, (school_name, (url_name, year_min, year_max, years_to_scrape)) in enumerate(teams_to_scrape.items(), 1):
        print(f"[{idx}/{len(teams_to_scrape)}] Processing {school_name}...")
        
        school_scraped = False
        for year in years_to_scrape:
            # Check if already exists (in case it was just created)
            output_file = os.path.join(defense_dir, f"{school_name.replace(' ', '_')}_{year}_defense.csv")
            if os.path.exists(output_file):
                print(f"  {year}: Already exists, skipping.")
                continue
            
            # Build URL using the correct URL name
            url = f"https://www.sports-reference.com/cfb/schools/{url_name}/{year}.html#all_defense_standard"
            
            print(f"  {year}: Scraping...", end=" ", flush=True)
            
            soup = get_soup(url)
            if not soup:
                print("Failed to fetch")
                time.sleep(5)
                continue
            
            try:
                # Try both table ID patterns (may vary by year)
                table_html = None
                for table_id in ['defense_standard', 'all_defense_standard']:
                    table_html = extract_table_from_comments(soup, table_id)
                    if table_html:
                        break
                
                if not table_html:
                    # Try to find it directly on the page with both IDs
                    for table_id in ['defense_standard', 'all_defense_standard']:
                        table = soup.find('table', id=table_id)
                        if table:
                            table_html = str(table)
                            break
                
                if not table_html:
                    print("No defense table found.")
                    time.sleep(5)
                    continue
                
                # Parse the table
                table_soup = BeautifulSoup(table_html, 'lxml')
                table = table_soup.find('table')
                
                if not table:
                    print("Could not parse table.")
                    time.sleep(5)
                    continue
                
                # Extract data from table
                data = []
                tbody = table.find('tbody')
                
                if tbody:
                    rows = tbody.find_all('tr')
                else:
                    rows = table.find_all('tr')[1:]
                
                for row in rows:
                    # Skip rows without data cells
                    cells = row.find_all('td')
                    if not cells:
                        continue
                    
                    row_data = {}
                    for cell in cells:
                        data_stat = cell.get('data-stat', '')
                        
                        # Extract player info from name_display column (similar to passing stats)
                        if data_stat == 'name_display':
                            a = cell.find('a')
                            if a and 'href' in a.attrs:
                                link = a['href']
                                name = a.get_text(strip=True).rstrip('*')
                                player_id = cell.get('data-append-csv', '')
                                row_data['Player'] = name
                                row_data['Player_ID'] = player_id
                                row_data['Player_Link'] = link
                            else:
                                row_data['Player'] = cell.get_text(strip=True).rstrip('*')
                                row_data['Player_ID'] = ''
                                row_data['Player_Link'] = ''
                        else:
                            row_data[data_stat] = cell.get_text(strip=True)
                    
                    data.append(row_data)
                
                if not data:
                    print("No data rows found.")
                    time.sleep(5)
                    continue
                
                # Create DataFrame and clean up
                df = pd.DataFrame(data)
                
                # Remove header rows if repeated
                if 'Rk' in df.columns:
                    df = df[df['Rk'] != 'Rk']
                
                # Add metadata
                df['School'] = school_name
                df['Year'] = year
                
                # Save to CSV
                df.to_csv(output_file, index=False)
                print(f"Saved ({len(df)} records)")
                scraped_count += 1
                school_scraped = True
                
            except Exception as e:
                print(f"Error parsing: {e}")
            
            # Rate limiting
            time.sleep(5)
        
        if not school_scraped:
            print(f"  No data scraped for this school.")
        print()  # Blank line for readability
    
    print(f"Defense stats scraping complete! Scraped {scraped_count} files.")

# Run the defense stats scraper - only scrape missing data
scrape_defense_stats(missing_only=True)

Loading school URL mappings...
Loaded 452 school URL mappings from X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\college_stats\HTML\school_list.txt

Checking for missing data...

Found 3 schools with missing data.

--- Starting defense stats scrape (missing_only=True) ---

[1/3] Processing Connecticut...
  2020: Scraping... No defense table found.
  No data scraped for this school.

[2/3] Processing Old Dominion...
  2020: Scraping... No defense table found.
  No data scraped for this school.

[3/3] Processing UAB...
  2015: Scraping... Error fetching https://www.sports-reference.com/cfb/schools/alabama-birmingham/2015.html#all_defense_standard: 404 Client Error: Not Found for url: https://www.sports-reference.com/cfb/schools/alabama-birmingham/2015.html#all_defense_standard
Failed to fetch
  2016: Scraping... Error fetching https://www.sports-reference.com/cfb/schools/alabama-birmingham/2016.html#all_defense_standard: 404 Client Error: Not Found for url: https://www.sports-referenc

In [16]:
# Debug: Check what's actually on a sports reference page
test_url = "https://www.sports-reference.com/cfb/schools/air-force/2025.html"
print(f"Fetching {test_url}...")

soup = get_soup(test_url)
if soup:
    print("\nAll tables on page:")
    all_tables = soup.find_all('table')
    for i, table in enumerate(all_tables):
        table_id = table.get('id', 'NO ID')
        print(f"  Table {i}: id='{table_id}'")
    
    print("\nLooking for 'defense' related content:")
    # Search for any mention of defense
    if 'defense' in soup.text.lower():
        print("  'defense' found in page text")
    
    # Check for comments
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    print(f"\n  Total comments found: {len(comments)}")
    defense_comments = [c for c in comments if 'defense' in c.lower()]
    print(f"  Comments containing 'defense': {len(defense_comments)}")
    
    if defense_comments:
        # Try to parse first defense comment
        print(f"\n  Parsing first defense comment...")
        for comm in defense_comments[:1]:
            comm_soup = BeautifulSoup(comm, 'lxml')
            tables_in_comm = comm_soup.find_all('table')
            print(f"    Tables in this comment: {len(tables_in_comm)}")
            for t in tables_in_comm:
                print(f"      - id: {t.get('id', 'NO ID')}")


Fetching https://www.sports-reference.com/cfb/schools/air-force/2025.html...

All tables on page:
  Table 0: id='team'
  Table 1: id='passing_standard'
  Table 2: id='rushing_standard'
  Table 3: id='kick_return_standard'

Looking for 'defense' related content:
  'defense' found in page text

  Total comments found: 76
  Comments containing 'defense': 1

  Parsing first defense comment...
    Tables in this comment: 1
      - id: defense_standard


In [ ]:
def check_defense_data_completeness(excel_path=None):
    """
    Validates that all schools have complete defense data for all years they were active.
    Reports any missing files.
    
    Args:
        excel_path: Path to Excel file. If None, uses default from BASE_DATA_DIR
    """
    print("\n--- Checking Defense Data Completeness ---\n")
    
    min_year = 2015
    max_year = 2025
    target_years = list(range(min_year, max_year + 1))
    
    # Load teams from Excel
    teams = load_teams_from_excel(excel_path, min_year, max_year)
    defense_dir = os.path.join(BASE_DATA_DIR, "Stats", "defense")
    
    missing_data = {}
    complete_schools = 0
    
    for school_name, url_name, year_min, year_max in teams:
        # Only check years this school was active
        active_years = [y for y in target_years if year_min <= y <= year_max]
        
        if not active_years:
            continue
        
        missing_years = []
        for year in active_years:
            file_path = os.path.join(defense_dir, f"{school_name.replace(' ', '_')}_{year}_defense.csv")
            if not os.path.exists(file_path):
                missing_years.append(year)
        
        if missing_years:
            missing_data[school_name] = missing_years
        else:
            complete_schools += 1
    
    # Report results
    print(f"Total schools checked: {len(teams)}")
    print(f"Complete schools: {complete_schools}")
    print(f"Schools with missing data: {len(missing_data)}\n")
    
    if missing_data:
        print("Schools missing data:")
        for school_name in sorted(missing_data.keys()):
            missing_years = missing_data[school_name]
            print(f"  {school_name}: Missing years {missing_years}")
    else:
        print("✓ All schools have complete defense data!")
    
    return missing_data

# Run the completeness check
missing = check_defense_data_completeness()


--- Checking Defense Data Completeness ---

Total schools checked: 137
Complete schools: 134
Schools with missing data: 3

Schools missing data:
  Connecticut: Missing years [2020]
  Old Dominion: Missing years [2020]
  UAB: Missing years [2015, 2016]


In [ ]:

# ============================================================================
# ROSTER SCRAPER - Scrapes full team rosters for all schools and years
# ============================================================================

def scrape_team_rosters(excel_path=None, missing_only=True):
    """
    Scrapes team rosters from Sports Reference for all schools across all eligible years.
    
    URL pattern: https://www.sports-reference.com/cfb/schools/{school-url}/{year}-roster.html
    
    Args:
        excel_path: Path to Excel file with team information. If None, uses default from BASE_DATA_DIR
        missing_only: If True, only scrape years/schools with missing roster data
    """
    
    min_year = 2015
    max_year = 2025
    target_years = list(range(min_year, max_year + 1))
    
    # Load school URL mapping from HTML
    print("Loading school URL mappings...")
    school_url_map = scrape_school_url_mapping()
    print()
    
    # Load teams from Excel
    teams = load_teams_from_excel(excel_path, min_year, max_year)
    roster_dir = os.path.join(BASE_DATA_DIR, "Stats", "rosters")
    
    if not os.path.exists(roster_dir):
        os.makedirs(roster_dir)
    
    # Update team URLs using the mapping
    teams_with_correct_urls = []
    for school_name, url_name, year_min, year_max in teams:
        correct_url_name = get_school_name_to_url(school_name, school_url_map)
        teams_with_correct_urls.append((school_name, correct_url_name, year_min, year_max))
    
    # Always check for existing files to skip them
    print("Checking for missing roster data...\n")
    teams_to_scrape = {}
    
    for school_name, url_name, year_min, year_max in teams_with_correct_urls:
        active_years = [y for y in target_years if year_min <= y <= year_max]
        if not active_years:
            continue
        
        missing_years = []
        for year in active_years:
            file_path = os.path.join(roster_dir, f"{school_name.replace(' ', '_')}_{year}_roster.csv")
            if not os.path.exists(file_path):
                missing_years.append(year)
        
        # Add to teams_to_scrape only if there are missing years (respects missing_only flag)
        if missing_years and missing_only:
            teams_to_scrape[school_name] = (url_name, year_min, year_max, missing_years)
        elif not missing_only:
            # When missing_only=False, include school but only scrape missing years anyway
            if missing_years:
                teams_to_scrape[school_name] = (url_name, year_min, year_max, missing_years)
    
    total_to_scrape = sum(len(years) for _, (_, _, _, years) in teams_to_scrape.items())
    print(f"Found {len(teams_to_scrape)} schools with {total_to_scrape} missing roster files to scrape.\n")
    
    print(f"--- Starting roster scrape (missing_only={missing_only}) ---\n")
    
    scraped_count = 0
    total_roster_records = 0
    
    for idx, (school_name, (url_name, year_min, year_max, years_to_scrape)) in enumerate(teams_to_scrape.items(), 1):
        print(f"[{idx}/{len(teams_to_scrape)}] Processing {school_name}...")
        
        school_scraped = False
        for year in years_to_scrape:
            # Check if already exists (in case it was just created)
            output_file = os.path.join(roster_dir, f"{school_name.replace(' ', '_')}_{year}_roster.csv")
            if os.path.exists(output_file):
                print(f"  {year}: Already exists, skipping.")
                continue
            
            # Build URL
            url = f"https://www.sports-reference.com/cfb/schools/{url_name}/{year}-roster.html"
            
            print(f"  {year}: Scraping...", end=" ", flush=True)
            
            soup = get_soup(url)
            if not soup:
                print("Failed to fetch")
                time.sleep(5)
                continue
            
            try:
                # Find the roster table (usually has id "roster")
                table = None
                
                # Try common table IDs for rosters
                for table_id in ['roster', 'all_roster']:
                    table_html = extract_table_from_comments(soup, table_id)
                    if table_html:
                        table_soup = BeautifulSoup(table_html, 'lxml')
                        table = table_soup.find('table')
                        if table:
                            break
                
                # If not found in comments, try direct search
                if not table:
                    for table_id in ['roster', 'all_roster']:
                        table = soup.find('table', id=table_id)
                        if table:
                            break
                
                if not table:
                    print("No roster table found.")
                    time.sleep(5)
                    continue
                
                # Extract data from table
                data = []
                tbody = table.find('tbody')
                
                if tbody:
                    rows = tbody.find_all('tr')
                else:
                    rows = table.find_all('tr')[1:]  # skip header
                
                for row in rows:
                    # Skip header rows (they have class="thead")
                    if row.get('class') and 'thead' in row.get('class', []):
                        continue
                    
                    row_data = {}
                    player_found = False
                    
                    # First, extract player info from the TH element with scope="row"
                    player_th = row.find('th', {'scope': 'row', 'data-stat': 'player'})
                    
                    if player_th:
                        a = player_th.find('a')
                        if a and 'href' in a.attrs:
                            link = a['href']
                            name = a.get_text(strip=True).rstrip('*')
                            # Extract player ID from href: /cfb/players/{player-id}.html
                            player_id_match = re.search(r'/cfb/players/([^/]+)\.html', link)
                            player_id = player_id_match.group(1) if player_id_match else ''
                            row_data['Player'] = name
                            row_data['Player_ID'] = player_id
                            row_data['Player_Link'] = link
                            player_found = True
                        else:
                            row_data['Player'] = player_th.get_text(strip=True).rstrip('*')
                            row_data['Player_ID'] = ''
                            row_data['Player_Link'] = ''
                            player_found = True
                    
                    # Skip if no player found
                    if not player_found:
                        continue
                    
                    # Now extract all TD cells for other attributes
                    cells = row.find_all('td')
                    for cell in cells:
                        data_stat = cell.get('data-stat', '')
                        cell_text = cell.get_text(strip=True)
                        if data_stat:
                            row_data[data_stat] = cell_text
                        else:
                            # If no data-stat, use a generic column name
                            row_data[f'col_{len(row_data)}'] = cell_text
                    
                    # Add row to data
                    data.append(row_data)
                
                if not data:
                    print("No roster data found.")
                    time.sleep(5)
                    continue
                
                # Create DataFrame
                df = pd.DataFrame(data)
                
                # Remove header rows if repeated
                if 'Rk' in df.columns:
                    df = df[df['Rk'] != 'Rk']
                
                # Add metadata
                df['School'] = school_name
                df['Year'] = year
                
                # Save to CSV
                df.to_csv(output_file, index=False)
                print(f"Saved ({len(df)} players)")
                scraped_count += 1
                total_roster_records += len(df)
                school_scraped = True
                
            except Exception as e:
                print(f"Error parsing: {e}")
            
            # Rate limiting
            time.sleep(5)
        
        if not school_scraped:
            print(f"  No roster data scraped for this school.")
        print()  # Blank line for readability
    
    print(f"Roster scraping complete!")
    print(f"Scraped {scraped_count} roster files")
    print(f"Total player records: {total_roster_records}")

# Run the roster scraper - only scrape missing data
print("=" * 70)
print("TEAM ROSTER SCRAPER FOR COLLEGE FOOTBALL")
print("=" * 70)
print()

scrape_team_rosters(missing_only=True)

TEAM ROSTER SCRAPER FOR COLLEGE FOOTBALL

Loading school URL mappings...
Loaded 452 school URL mappings from X:\My Files\Courses\DSBA 6010 - LLM\Project\Code\college_stats\HTML\school_list.txt

Checking for missing roster data...

Found 137 schools with 1442 missing roster files to scrape.

--- Starting roster scrape (missing_only=True) ---

[1/137] Processing Air Force...
  2015: Scraping... Saved (87 players)
  2016: Scraping... Saved (50 players)
  2017: Scraping... Saved (99 players)
  2018: Scraping... Saved (99 players)
  2019: Scraping... Saved (109 players)
  2020: Scraping... Saved (119 players)
  2021: Scraping... Saved (115 players)
  2022: Scraping... Saved (134 players)
  2023: Scraping... Saved (125 players)
  2024: Scraping... Saved (136 players)
  2025: Scraping... Saved (141 players)

[2/137] Processing Akron...
  2015: Scraping... Saved (108 players)
  2016: Scraping... Saved (57 players)
  2017: Scraping... Saved (102 players)
  2018: Scraping... Saved (109 players)


In [19]:
def test_roster_scraper(school='air-force', year=2015):
    """
    Test the roster scraper on a single school/year to verify player extraction works correctly.
    
    Args:
        school: URL name of school (default: 'air-force')
        year: Year to test (default: 2015)
    """
    print(f"\n=== Testing Roster Scraper ===")
    print(f"School: {school}")
    print(f"Year: {year}\n")
    
    url = f"https://www.sports-reference.com/cfb/schools/{school}/{year}-roster.html"
    print(f"Fetching: {url}\n")
    
    soup = get_soup(url)
    if not soup:
        print("ERROR: Failed to fetch page")
        return
    
    # Find the roster table
    table = None
    for table_id in ['roster', 'all_roster']:
        table_html = extract_table_from_comments(soup, table_id)
        if table_html:
            table_soup = BeautifulSoup(table_html, 'lxml')
            table = table_soup.find('table')
            if table:
                print(f"✓ Found table with id='{table_id}'")
                break
    
    if not table:
        for table_id in ['roster', 'all_roster']:
            table = soup.find('table', id=table_id)
            if table:
                print(f"✓ Found table with id='{table_id}'")
                break
    
    if not table:
        print("ERROR: No roster table found")
        return
    
    # Extract rows
    tbody = table.find('tbody')
    rows = tbody.find_all('tr') if tbody else table.find_all('tr')[1:]
    
    print(f"✓ Found {len(rows)} total rows in table")
    print()
    
    # Process rows
    valid_rows = 0
    players_extracted = []
    
    for idx, row in enumerate(rows):
        # Skip header rows
        if row.get('class') and 'thead' in row.get('class', []):
            print(f"Row {idx}: SKIPPED (header row)")
            continue
        
        # Try to extract player info
        player_th = row.find('th', {'scope': 'row', 'data-stat': 'player'})
        
        if not player_th:
            print(f"Row {idx}: SKIPPED (no player TH element)")
            continue
        
        # Extract name and ID
        a = player_th.find('a')
        if not a or 'href' not in a.attrs:
            print(f"Row {idx}: SKIPPED (no player link)")
            continue
        
        link = a['href']
        name = a.get_text(strip=True).rstrip('*')
        player_id_match = re.search(r'/cfb/players/([^/]+)\.html', link)
        player_id = player_id_match.group(1) if player_id_match else ''
        
        # Extract other attributes
        cells = row.find_all('td')
        attributes = {}
        for cell in cells:
            data_stat = cell.get('data-stat', '')
            if data_stat:
                attributes[data_stat] = cell.get_text(strip=True)
        
        players_extracted.append({
            'name': name,
            'player_id': player_id,
            'class': attributes.get('class', ''),
            'pos': attributes.get('pos', ''),
            'summary': attributes.get('summary', '')
        })
        
        valid_rows += 1
        if valid_rows <= 5:  # Print first 5
            print(f"Row {idx}: ✓ Player '{name}' (ID: {player_id})")
            print(f"  Class: {attributes.get('class')}, Pos: {attributes.get('pos')}")
    
    print()
    print(f"=== Results ===")
    print(f"Total rows processed: {len(rows)}")
    print(f"Valid player rows extracted: {valid_rows}")
    print(f"Extraction rate: {100*valid_rows/len(rows):.1f}%")
    
    if players_extracted:
        print(f"\nSample extracted data:")
        for p in players_extracted[:3]:
            print(f"  • {p['name']:25s} | ID: {p['player_id']:20s} | {p['pos']:3s}")
    
    return players_extracted

# Run the test
test_data = test_roster_scraper('air-force', 2015)



=== Testing Roster Scraper ===
School: air-force
Year: 2015

Fetching: https://www.sports-reference.com/cfb/schools/air-force/2015-roster.html

✓ Found table with id='roster'
✓ Found 87 total rows in table

Row 0: ✓ Player 'Karson Roberts' (ID: karson-roberts-1)
  Class: SR, Pos: QB
Row 1: ✓ Player 'Nate Romine' (ID: nate-romine-1)
  Class: JR, Pos: QB
Row 2: ✓ Player 'Ryan Brand' (ID: ryan-brand-1)
  Class: FR, Pos: QB
Row 3: ✓ Player 'Pate Davis' (ID: pate-davis-2)
  Class: JR, Pos: QB
Row 4: ✓ Player 'Jacobi Owens' (ID: jacobi-owens-1)
  Class: JR, Pos: RB

=== Results ===
Total rows processed: 87
Valid player rows extracted: 87
Extraction rate: 100.0%

Sample extracted data:
  • Karson Roberts            | ID: karson-roberts-1     | QB 
  • Nate Romine               | ID: nate-romine-1        | QB 
  • Ryan Brand                | ID: ryan-brand-1         | QB 
